# ERTInv - synthetic ERT modelling & inversion with pyGIMLi

**Electrical Resistivity Tomography (ERT)** images the subsurface by injecting current
through electrode pairs and measuring the resulting voltages. Each measurement senses a
weighted average of the ground's resistivity; from many measurements an **inversion**
reconstructs a 2-D resistivity section.

This notebook runs a **fully synthetic experiment**, which is the standard way to test and
understand an inversion:

1. we **build a known ('true') geological model**,
2. **simulate** the ERT data it would produce (forward modelling), with added noise,
3. **invert** that data as if we didn't know the answer,
4. **compare** the recovered model with the truth.

Because we know the true model, we can judge honestly how faithfully - and how deep - the
inversion recovers it, and how that degrades with noise.

Everything you control lives in **section 2**. One run uses **one** inversion method and
writes a single 4-page PDF report. To try another method, change one line and re-run.


## 1. Setup

This makes the `ertinv` package importable (it lives one folder up from this notebook)
and pulls in the objects you'll use. Nothing scientific happens here.

`%autoreload` makes Jupyter pick up any edit to the `ertinv/*.py` files without a manual
kernel restart - handy while developing.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys

# make the src package importable no matter where the kernel starts
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src import (
    Config, Layer, Target, Survey, Forward, Inversion,
    default_config, run, compare_methods, compare_arrays,
)
print("src package ready. Repo root:", REPO_ROOT)

## 2. Describe the experiment

This is the control panel. Everything you can change lives here - nothing is hidden inside
the processing code.

### The geological model
`default_config()` builds the original ERTInv example: **three horizontal layers**
(100 / 80 / 200 Ohm-m from top to bottom) with **three buried targets** - a resistive block,
a conductive block and a large deep resistor. You can edit these, or build a completely new
model in section 5.

### The inversion method (pick ONE)
All four are *genuine* pyGIMLi regularisation schemes, not just different `lambda` values.
Regularisation is what makes the ill-posed inverse problem solvable - it decides *which* of
the many models fitting the data you actually get:

- **`occam`** - smooth, first-order roughness. Penalises sharp contrasts, so it returns the
  smoothest model consistent with the data. The safe, standard choice; blurs sharp boundaries.
- **`marquardt`** - zeroth-order (Marquardt-Levenberg) damping. Pulls the model toward a
  homogeneous reference. Classic for few-parameter problems; tends to under-fit dense 2-D
  pixel models, so expect a higher chi2.
- **`blocky`** - L1 norm on the model roughness. Allows a few **sharp jumps**, so it recovers
  block-like bodies with crisp edges (good for walls, ore bodies, UXO-like targets).
- **`robust`** - L1 norm on the data misfit. Down-weights **outliers**, so a few bad readings
  don't drag the whole model. Use it when the data are noisy or spiky.

### Noise levels
Each value is the relative Gaussian noise added to the simulated data (0.05 = 5%). The
pipeline simulates and inverts one dataset per level, so you see how the result degrades.


In [ ]:
cfg = default_config()          # 3 layers + 3 buried targets, 31-electrode dipole-dipole

# --- inversion method (pick ONE): occam | marquardt | blocky | robust
cfg.inversion.method = "occam"

# --- noise levels to simulate & invert -------------------------------
cfg.forward.noise_levels = [0.02, 0.05, 0.10]     # 2%, 5%, 10%

# --- where the PDF report goes ---------------------------------------
cfg.outdir = os.path.join(REPO_ROOT, "results")
cfg.datadir = os.path.join(REPO_ROOT, "data")     # simulated .dat files (kept out of results/)

print(cfg.describe())

## 3. Run the pipeline

One call does the whole chain. Under the hood:

1. **Build the true model** on a fine, geometry-conforming mesh.
2. **Forward modelling** - solve the electrical problem on that mesh to get the apparent
   resistivities the survey would measure, then add the requested noise. This is done on a
   *different* mesh from the one the inversion uses, to avoid the **'inverse crime'**
   (artificially good fits from inverting on the very grid that produced the data).
3. **Inversion** - for each noise level, iteratively adjust a model on an independent
   parameter mesh until its predicted data match the observations to within the noise
   (Gauss-Newton steps with the chosen regularisation).
4. **Metrics** - data misfit (chi2 / RMS) and, since we know the truth, the model-space
   error (correlation between recovered and true resistivity).

**Output:** a single 4-page PDF (`results/ERTInv_report_<method>.pdf`):
true model / pseudosections / recovered models / residual histograms. Nothing is drawn
inline; the last cell opens the PDF for you.


In [ ]:
results = run(cfg)          # quiet; writes ONE 4-page pdf

print()
print(results.rms_table())

### How to read the numbers

`chi2` is the noise-weighted data misfit, `chi2 = mean( ((d_obs - d_pred) / sigma)^2 )`,
and here `RMS = sqrt(chi2)`.

- **chi2 approx 1** is the target: the model fits the data *to within the noise* - neither
  under- nor over-fitting.
- **chi2 >> 1**: under-fit - the model is too smooth / wrong; the data are not explained.
- **chi2 << 1**: over-fit - the model is chasing noise and inventing structure.

A healthy run sits close to 1 at every noise level, rising gently as noise grows. The
**model correlation** (printed per level) is a luxury only synthetic tests have: it measures
how much the recovered section actually looks like the truth, and naturally drops as noise
increases.


In [ ]:
# open the PDF report (Windows)
report = os.path.join(cfg.outdir, f"ERTInv_report_{cfg.inversion.method}.pdf")
print("Report:", report)
try:
    os.startfile(report)
except Exception:
    print("(open it manually from the path above)")

## 4. Compare all four methods (optional)

Runs occam / marquardt / blocky / robust on a **single** noise level and writes a
side-by-side figure (`results/method_comparison.png`). This is the quickest way to *see*
what each regularisation does to the same data: how occam blurs, how blocky sharpens, and
how their chi2 values differ. A lower chi2 is not automatically 'better' - a method can fit
the data well while placing structure in the wrong place, which is exactly why the true
model matters.


In [ ]:
compare_methods(cfg, methods=["occam", "marquardt", "blocky", "robust"], noise_level=0.05)

## 5. Build your own model from scratch (optional)

Replace the default geology with your own. A `Config` has five parts:

- **`layers`** - top-to-bottom horizontal layers; the last one has `bottom=None` (it fills
  the rest of the world). `resistivity` in Ohm-m.
- **`targets`** - buried bodies as polygons (`points` = corner coordinates in metres,
  x horizontal, z negative downward), each with its own `resistivity`.
- **`survey`** - electrode array (`dd`, `wa`, `slm`, ...), number of electrodes and extent.
- **`forward`** - the noise levels to simulate.
- **`inversion`** - the method to use.

The example below is a shallow conductive block in a three-layer background, inverted with
`blocky` to recover its sharp edges.


In [ ]:
my_cfg = Config(
    layers=[
        Layer(bottom=-3.0,  resistivity=100.0, name='topsoil'),
        Layer(bottom=-10.0, resistivity=50.0,  name='clay'),
        Layer(bottom=None,  resistivity=300.0, name='bedrock'),
    ],
    targets=[
        Target(name='buried block', resistivity=15.0, area=0.4,
               points=[[-4, -2], [4, -2], [4, -6], [-4, -6]]),
    ],
    survey=Survey(scheme='dd', n_electrodes=31, x_start=-20, x_end=20),
    forward=Forward(noise_levels=[0.03, 0.07]),
    inversion=Inversion(method='blocky'),
    outdir=os.path.join(REPO_ROOT, 'results_custom'),
    datadir=os.path.join(REPO_ROOT, 'data'),
)
run(my_cfg)